# Working with Containers

This notebook is a guided tour of the main data containers provided by `python-blosc2`.

The goal is to build a practical mental model first: what each container is, how it is implemented, what features it provides, and when it is the right tool for the job.

We will cover these containers in this order:

1. `SChunk`
2. `NDArray`
3. `VLArray`
4. `BatchStore`
5. `EmbedStore`
6. `DictStore`
7. `TreeStore`
8. `C2Array`

In [ ]:
def show(label, value):
    print(f"{label}: {value}")

## The Big Picture

`SChunk` is the storage foundation. Higher-level containers either wrap it to provide a more convenient programming model, or use it as a building block inside larger stores.

- `NDArray` adds N-dimensional array semantics on top of chunked compressed storage.
- `VLArray` stores one variable-length serialized item per entry.
- `BatchStore` stores batches of variable-length items.
- `EmbedStore`, `DictStore`, and `TreeStore` organize multiple containers together.
- `C2Array` is different: it is a remote array handle rather than a local storage container.

<img src="images/containers/overview.svg" width="1000"/>

Later iterations of this notebook will replace the remaining placeholders with small SVG diagrams.

## `SChunk`: The Foundation

`SChunk` is the low-level compressed storage container in Blosc2. Conceptually, it is a sequence of compressed chunks plus metadata.

It is useful when you want direct control over chunk-oriented storage, chunk append/update operations, or persistent compressed payloads without array semantics.

Implementation notes:

- stores data as compressed chunks
- can live in memory or on disk
- supports metadata and variable-length metadata
- acts as the basis for higher-level containers such as `NDArray`, `VLArray`, and `BatchStore`

> Figure placeholder: `images/containers/schunk.svg`

In [ ]:
# TODO: add a compact SChunk example.
# Suggested topics:
# - create an empty SChunk
# - append a couple of chunks
# - inspect nchunks, nbytes, cbytes, and metadata
# - optionally reopen from disk

## `NDArray`: Compressed N-D Arrays

`NDArray` is the main dense-array container in `python-blosc2`. It adds array semantics such as shape, dtype, slicing, chunking, and persistence on top of an underlying `SChunk`.

It is useful for dense numeric data when you want array operations together with compressed storage.

Implementation notes:

- built on top of `SChunk`
- stores shape/chunk/block layout as array metadata
- supports in-memory and persistent arrays
- integrates with slicing and higher-level compute workflows

> Figure placeholder: `images/containers/ndarray.svg`

In [ ]:
# TODO: add a compact NDArray example.
# Suggested topics:
# - create a small array
# - persist it to disk
# - show shape, chunks, and blocks
# - read a small slice

## `VLArray`: Variable-Length Items

`VLArray` is a list-like container for variable-length Python values. Each entry is serialized and stored as its own compressed chunk in a backing `SChunk`.

It is useful for ragged or heterogeneous values such as strings, dictionaries, tuples, lists, and byte payloads.

Implementation notes:

- backed by a single `SChunk`
- each logical entry is stored independently
- uses serialization before compression
- offers list-like indexing, insertion, deletion, and persistence

> Figure placeholder: `images/containers/vlarray.svg`

In [ ]:
# TODO: add a compact VLArray example.
# Suggested topics:
# - append variable-length values
# - read entries and slices
# - show that values can have different sizes

## `BatchStore`: Batched Variable-Length Data

`BatchStore` is designed for batch-oriented variable-length data. Instead of storing one item per chunk, it stores one batch per chunk, with optional internal subdivision for more efficient item access inside a batch.

It is useful when data arrives or is processed in batches and batch-level append/update operations are the natural API.

Implementation notes:

- backed by a single `SChunk`
- each chunk stores one batch
- batches may contain multiple internal variable-length blocks
- indexing the store returns batches; a `Batch` object acts as the lazy per-batch view

> Figure placeholder: `images/containers/vlarray-batchstore.svg`

In [ ]:
# TODO: add a compact BatchStore example.
# Suggested topics:
# - append a few batches
# - index by batch
# - iterate items with iter_items()
# - inspect summary info

## `EmbedStore`: Bundle Several Containers Into One Store

`EmbedStore` is a dictionary-like container that stores several Blosc2 objects as embedded nodes inside one backing store.

It is useful when you want to package several arrays or container objects into one portable object or file.

Implementation notes:

- stores serialized nodes inside one internal backing container
- can embed arrays and chunked containers together
- may keep lightweight references for remote `C2Array` entries
- focuses on compact bundling rather than hierarchical organization

> Figure placeholder: `images/containers/embedstore.svg`

In [ ]:
# TODO: add a compact EmbedStore example.
# Suggested topics:
# - create a store
# - add a small NDArray and SChunk-like object
# - retrieve them back

## `DictStore`: Key-Value Collection of Containers

`DictStore` is a directory- or zip-backed key-value collection for Blosc2 objects.

It is useful when you want to organize a dataset made of several named arrays or containers, while keeping storage portable.

Implementation notes:

- manages multiple objects under string keys
- can use embedded storage for small objects and external leaves for larger ones
- supports `.b2d` directory stores and `.b2z` zip stores
- is the non-hierarchical basis for `TreeStore`

> Figure placeholder: `images/containers/dictstore.svg`

In [ ]:
# TODO: add a compact DictStore example.
# Suggested topics:
# - create a .b2d or .b2z store
# - assign a couple of named entries
# - list the keys and reopen the store

## `TreeStore`: Hierarchical Datasets

`TreeStore` extends `DictStore` with hierarchical keys such as `/group/subgroup/node`.

It is useful when your data naturally forms a tree, and you want subtree traversal or path-based organization.

Implementation notes:

- built on top of `DictStore`
- validates and manages hierarchical paths
- supports subtree traversal and structural organization
- keeps the same storage ideas as `DictStore`, but with a tree model

> Figure placeholder: `images/containers/stores.svg`

In [ ]:
# TODO: add a compact TreeStore example.
# Suggested topics:
# - assign arrays under hierarchical paths
# - walk a subtree
# - show the difference from DictStore

## `C2Array`: Remote Arrays

`C2Array` is a remote array handle for Caterva2-hosted arrays. Unlike the local containers above, it does not primarily manage local storage; instead, it exposes remote metadata and remote slice access.

It is useful when the array lives on a remote service and you want to read metadata or fetch slices without managing the full local dataset yourself.

Implementation notes:

- acts as a remote handle rather than a local chunk store
- fetches metadata and slices over HTTP
- can be referenced by local higher-level stores such as `EmbedStore`
- is best presented as the remote-facing member of the container family

> Figure placeholder: `images/containers/c2array.svg`

In [ ]:
# TODO: add a compact C2Array example.
# Suggested topics:
# - open a small remote array
# - inspect metadata
# - fetch a tiny slice
# Note: this cell may require network access.

## Choosing the Right Container

| Container | Implemented on top of | Main strengths | Best for |
| --- | --- | --- | --- |
| `SChunk` | native chunk container | direct chunk control, metadata, persistence | chunk-oriented storage |
| `NDArray` | `SChunk` | array semantics, slicing, persistence | dense numeric arrays |
| `VLArray` | `SChunk` | variable-length entries | ragged or heterogeneous items |
| `BatchStore` | `SChunk` | batch append/update, item access inside batches | batched variable-length data |
| `EmbedStore` | internal backing store | compact bundling of several objects | portable bundles |
| `DictStore` | `EmbedStore` + external leaves | named collections | multi-object datasets |
| `TreeStore` | `DictStore` | hierarchy and subtree traversal | structured datasets |
| `C2Array` | remote service | remote metadata and slicing | remote arrays |

This table should stay near the end of the notebook, after the reader has already seen each container in context.

## Final Notes

This notebook is intentionally organized from low-level storage to higher-level organization:

- start with `SChunk` to understand the storage basis
- move to array and variable-length abstractions built on top of it
- finish with collection-oriented stores and remote access

Next iterations should replace the placeholders with:

- polished short explanations
- runnable compact examples
- SVG diagrams under `images/containers/`
- small transitions between sections so the narrative reads continuously